In [2]:
%pip install catboost lightgbm optuna holidays

In [3]:
import numpy as np
import pandas as pd
import duckdb

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

import holidays



In [4]:
def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

DRIVE_FOLDER_NAME = "Store Sales"

In [5]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [6]:
TARGET = "sales"
N_SPLITS = 5
SEED = 42
eps = 1e-5

def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

non_relevant_columns = ['id', 'source']

# TODO: Add categorical columns of original dataframes
# FIXME: FIX test
# ordinal_maps = {
#     "gender" : {"male":0, "female":1, "other":2},
#     "internet_access" : {"no":0, "yes":1},
#     "sleep_quality" : {"poor":0, "average":1, "good":2},
#     "facility_rating" : {"low":0, "medium":1, "high":2},
#     "exam_difficulty" : {"easy":0, "moderate":1, "hard":2},
#     "course" : {"ba":0, "b.sc":1, "diploma":2, "b.tech":3, "b.com":4, "bca":5, "bba":6},
#     "study_method" : {"self-study":0, "online videos":1, "group study":2, "mixed":3, "coaching":4},
# }

# original_categorical_cols = ['country', 'store', 'product']


In [7]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/train.csv')
    test_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  train_df['date'] =  pd.to_datetime(train_df['date'])
  test_df['date'] =  pd.to_datetime(test_df['date'])
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

In [8]:
def load_helper_data(option: str="local"):
  '''
  returns oil_df, transactions_df, holidays_events_df, stores_df

  '''
  if option == "local":
    oil_df = pd.read_csv('oil.csv')
    transactions_df = pd.read_csv('transactions.csv')
    holidays_events_df = pd.read_csv('holidays_events.csv')
    stores_df = pd.read_csv('stores.csv')


  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    oil_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/oil.csv')
    transactions_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/transactions.csv')
    holidays_events_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/holidays_events.csv')
    stores_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/stores.csv')



  return oil_df, transactions_df, holidays_events_df, stores_df

In [9]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

In [10]:
train_df, test_df, df = load_data(ENVIRONMENT)
oil_df, transactions_df, holidays_events_df, stores_df = load_helper_data(ENVIRONMENT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# EDA

In [17]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,source
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,train
1,1,2013-01-01,1,BABY CARE,0.0,0,train
2,2,2013-01-01,1,BEAUTY,0.0,0,train
3,3,2013-01-01,1,BEVERAGES,0.0,0,train
4,4,2013-01-01,1,BOOKS,0.0,0,train


In [13]:
df.groupby('source').agg(min_date=('date', 'min'), max_date=('date', 'max'))

,min_date,max_date
source,,
test,2017-08-16,2017-08-31
train,2013-01-01,2017-08-15


In [14]:
oil_df.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [15]:
transactions_df.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [19]:
holidays_events_df

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
...,...,...,...,...,...,...
345,2017-12-22,Additional,National,Ecuador,Navidad-3,False
346,2017-12-23,Additional,National,Ecuador,Navidad-2,False
347,2017-12-24,Additional,National,Ecuador,Navidad-1,False
348,2017-12-25,Holiday,National,Ecuador,Navidad,False


In [28]:
holidays_events_df['locale_name'].unique()

array(['Manta', 'Cotopaxi', 'Cuenca', 'Libertad', 'Riobamba', 'Puyo',
       'Guaranda', 'Imbabura', 'Latacunga', 'Machala', 'Santo Domingo',
       'El Carmen', 'Cayambe', 'Esmeraldas', 'Ecuador', 'Ambato',
       'Ibarra', 'Quevedo', 'Santo Domingo de los Tsachilas',
       'Santa Elena', 'Quito', 'Loja', 'Salinas', 'Guayaquil'],
      dtype=object)

In [18]:
stores_df.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [34]:
df.sort_values(by='date', ascending=True).reset_index(drop=True).head(10)

,id,date,store_nbr,family,sales,onpromotion,source
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,train
1,1194,2013-01-01,42,CELEBRATION,0.0,0,train
2,1193,2013-01-01,42,BREAD/BAKERY,0.0,0,train
3,1192,2013-01-01,42,BOOKS,0.0,0,train
4,1191,2013-01-01,42,BEVERAGES,0.0,0,train
5,1190,2013-01-01,42,BEAUTY,0.0,0,train
6,1189,2013-01-01,42,BABY CARE,0.0,0,train
7,1188,2013-01-01,42,AUTOMOTIVE,0.0,0,train
8,1187,2013-01-01,41,SEAFOOD,0.0,0,train
9,1186,2013-01-01,41,SCHOOL AND OFFICE SUPPLIES,0.0,0,train


In [36]:
df.index[df['date'] == '2017-01-01']

Index([2596374, 2596375, 2596376, 2596377, 2596378, 2596379, 2596380, 2596381,
       2596382, 2596383,
       ...
       2598146, 2598147, 2598148, 2598149, 2598150, 2598151, 2598152, 2598153,
       2598154, 2598155],
      dtype='int64', length=1782)

In [ ]:
dates = np.sort(df["date"].unique())

tscv = TimeSeriesSplit(n_splits=4, test_size=16)   # 16 days = competition horizon

for i in enumerate(tscv.split(dates)):
    print(i)

(0, (array([   0,    1,    2, ..., 1633, 1634, 1635]), array([1636, 1637, 1638, 1639, 1640, 1641, 1642, 1643, 1644, 1645, 1646,
       1647, 1648, 1649, 1650, 1651])))
(1, (array([   0,    1,    2, ..., 1649, 1650, 1651]), array([1652, 1653, 1654, 1655, 1656, 1657, 1658, 1659, 1660, 1661, 1662,
       1663, 1664, 1665, 1666, 1667])))
(2, (array([   0,    1,    2, ..., 1665, 1666, 1667]), array([1668, 1669, 1670, 1671, 1672, 1673, 1674, 1675, 1676, 1677, 1678,
       1679, 1680, 1681, 1682, 1683])))
(3, (array([   0,    1,    2, ..., 1681, 1682, 1683]), array([1684, 1685, 1686, 1687, 1688, 1689, 1690, 1691, 1692, 1693, 1694,
       1695, 1696, 1697, 1698, 1699])))


array(['2013-01-01T00:00:00.000000000', '2013-01-02T00:00:00.000000000',
       '2013-01-03T00:00:00.000000000', ...,
       '2017-08-29T00:00:00.000000000', '2017-08-30T00:00:00.000000000',
       '2017-08-31T00:00:00.000000000'], dtype='datetime64[ns]')

In [29]:
stores_df['state'].unique()

array(['Pichincha', 'Santo Domingo de los Tsachilas', 'Cotopaxi',
       'Chimborazo', 'Imbabura', 'Bolivar', 'Pastaza', 'Tungurahua',
       'Guayas', 'Santa Elena', 'Los Rios', 'Azuay', 'Loja', 'El Oro',
       'Esmeraldas', 'Manabi'], dtype=object)